In [ ]:
!pip install torch transformers pandas scikit-learn
!pip install 'git+https://github.com/SKTBrain/KoBERT.git#egg=kobert_tokenizer&subdirectory=kobert_hf'

  Cloning https://github.com/SKTBrain/KoBERT.git to /tmp/pip-install-x796vuv7/kobert-tokenizer_6168b7e82cc34b29b842c4f11c9a705d
  Running command git clone --filter=blob:none --quiet https://github.com/SKTBrain/KoBERT.git /tmp/pip-install-x796vuv7/kobert-tokenizer_6168b7e82cc34b29b842c4f11c9a705d
  Resolved https://github.com/SKTBrain/KoBERT.git to commit fcd729f2f4b37858f333597c0782388ada51eb5f
  Preparing metadata (setup.py) ... done
  Created wheel for kobert_tokenizer: filename=kobert_tokenizer-0.1-py3-none-any.whl size=4633 sha256=3431bdace109c69f965fb6b5eefbd8f088afb31157e2ef3cfd4594c2aeecca41
  Stored in directory: /tmp/pip-ephem-wheel-cache-vqjotux1/wheels/ec/ae/fb/30f74ad83a90c3f950522723b8d918d197fa77d6bd7df7b028
Successfully built kobert_tokenizer


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. 데이터 로드
try:
    df = pd.read_csv('train_data.csv', encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv('train_data.csv', encoding='euc-kr')

print(f"데이터 로드 완료: {len(df)}개 샘플")

# 2. 데이터 전처리 (결측치 제거)
df = df.dropna(subset=['title']) # 'title' 컬럼의 NaN 값 제거
df = df.dropna(subset=['topic_idx']) # 'topic_idx' 컬럼의 NaN 값 제거

# 3. 데이터 분리
X = df['title'] # 뉴스 기사 제목
y = df['topic_idx'] # 뉴스 주제

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. TF-IDF 벡터화
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 5. 모델 학습 및 평가
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)
y_pred = model.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)

print("--- 베이스라인(TF-IDF + 로지스틱 회귀) 결과 ---")
print(f'정확도: {accuracy * 100:.2f}%')

데이터 로드 완료: 45654개 샘플
--- 베이스라인(TF-IDF + 로지스틱 회귀) 결과 ---
정확도: 76.97%


In [ ]:
# =======================================================
# 1. 필수 라이브러리 설치
# =======================================================
print("===== 1. 라이브러리 설치 시작 =====")
!pip install torch transformers pandas scikit-learn tqdm
# KoBERT 토크나이저를 위한 설치
!pip install 'git+https://github.com/SKTBrain/KoBERT.git#egg=kobert_tokenizer&subdirectory=kobert_hf'
print("===== 1. 라이브러리 설치 완료 =====")


# =======================================================
# 2. 라이브러리 임포트 및 GPU 설정
# =======================================================
print("\n===== 2. 라이브러리 임포트 및 GPU 설정 =====")
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel
from torch.optim import AdamW
from kobert_tokenizer import KoBERTTokenizer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
from tqdm.notebook import tqdm # 진행률 표시

# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# =======================================================
# 3. 데이터 로드 및 전처리
# =======================================================
print("\n===== 3. 데이터 로드 및 전처리 =====")
# Colab에 업로드한 'train_data.csv' 파일 읽기
try:
    df = pd.read_csv('train_data.csv', encoding='utf-8')
except FileNotFoundError:
    print("\n*** 에러: 'train_data.csv' 파일을 Colab에 업로드했는지 확인하세요. ***\n")
    raise
except UnicodeDecodeError:
    df = pd.read_csv('train_data.csv', encoding='euc-kr')

# 결측치 제거
df = df.dropna(subset=['title', 'topic_idx'])

# [중요] 학습 시간 단축을 위해 10,000개 샘플만 사용
# (실제 프로젝트에서는 전체 데이터를 사용하거나 더 많이 샘플링하세요)
df = df.sample(n=10000, random_state=42)
print(f"로드된 샘플 수: {len(df)}")

# 데이터셋 분리 (Train: 80%, Test: 20%)
X = df['title']
y = df['topic_idx']
num_labels = len(y.unique()) # 7 (DACON 기준 7개 주제)
print(f"총 레이블 개수: {num_labels}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train 샘플: {len(X_train)}, Test 샘플: {len(X_test)}")


# =======================================================
# 4. KoBERT 토크나이저 및 커스텀 데이터셋 정의
# =======================================================
print("\n===== 4. 토크나이저 및 데이터셋 정의 =====")
# KoBERT 토크나이저 로드
tokenizer = KoBERTTokenizer.from_pretrained('skt/kobert-base-v1')

# PyTorch 커스텀 데이터셋 클래스
class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]

        # KoBERT 토크나이저로 텍스트 인코딩
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,    # [CLS], [SEP] 추가
            max_length=self.max_len,    # 최대 길이 64
            return_token_type_ids=False,
            padding='max_length',       # 패딩
            truncation=True,            # 잘라내기
            return_attention_mask=True, # 어텐션 마스크 반환
            return_tensors='pt',        # PyTorch 텐서로 반환
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

# 데이터셋 및 데이터로더 생성
train_dataset = NewsDataset(X_train, y_train, tokenizer)
test_dataset = NewsDataset(X_test, y_test, tokenizer)

BATCH_SIZE = 32 # 배치 사이즈
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print("데이터로더 생성 완료.")


# =======================================================
# 5. KoBERT 분류 모델 정의
# =======================================================
print("\n===== 5. KoBERT 분류 모델 정의 =====")
class KoBERTClassifier(nn.Module):
    def __init__(self, bert, num_labels):
        super(KoBERTClassifier, self).__init__()
        self.bert = bert
        self.dropout = nn.Dropout(p=0.1)
        # BERT 출력(768) -> 분류할 레이블 개수(num_labels)
        self.classifier = nn.Linear(768, num_labels)

    def forward(self, input_ids, attention_mask):
        # BERT 모델 실행
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        # [CLS] 토큰에 해당하는 pooler_output 사용
        pooled_output = outputs.pooler_output
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

# 모델 로드
bert_model = BertModel.from_pretrained('skt/kobert-base-v1')
model = KoBERTClassifier(bert_model, num_labels=num_labels).to(device)
print("모델 로드 완료.")


# =======================================================
# 6. 학습 및 평가 루프 정의
# =======================================================
print("\n===== 6. 학습 및 평가 루프 정의 =====")
# 옵티마이저 (AdamW) 및 손실 함수 (CrossEntropyLoss)
optimizer = AdamW(model.parameters(), lr=2e-5) # 2e-5가 BERT Fine-tuning에 권장됨
loss_fn = nn.CrossEntropyLoss().to(device)

# --- 학습 함수 ---
def train_epoch(model, data_loader, loss_fn, optimizer, device):
    model.train() # 학습 모드
    total_loss = 0

    for batch in tqdm(data_loader, desc="[Train]"):
        # 데이터를 GPU로 이동
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # 순전파
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fn(outputs, labels)
        total_loss += loss.item()

        # 역전파
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    return total_loss / len(data_loader)

# --- 평가 함수 ---
def eval_model(model, data_loader, loss_fn, device):
    model.eval() # 평가 모드
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad(): # 그래디언트 계산 비활성화
        for batch in tqdm(data_loader, desc="[Eval]"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()

            _, preds = torch.max(outputs, dim=1) # 가장 높은 확률의 인덱스

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    avg_loss = total_loss / len(data_loader)
    return accuracy, avg_loss

print("학습/평가 함수 정의 완료.")


# =======================================================
# 7. 모델 학습 실행
# =======================================================
print("\n===== 7. 모델 학습 시작 =====")
EPOCHS = 3 # 3 에포크 정도가 적당합니다. (GPU로 약 15~20분 소요)
final_val_acc = 0.0

for epoch in range(EPOCHS):
    print(f'--- Epoch {epoch + 1}/{EPOCHS} ---')
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer, device)
    print(f'Train Loss: {train_loss:.4f}')

    val_acc, val_loss = eval_model(model, test_loader, loss_fn, device)
    print(f'Val Loss: {val_loss:.4f}, Val Accuracy: {val_acc * 100:.2f}%')
    final_val_acc = val_acc # 마지막 에포크의 정확도를 저장

print("===== 학습 완료 =====")
print(f"*** 최종 테스트 정확도: {final_val_acc * 100:.2f}% ***")

===== 1. 라이브러리 설치 시작 =====
  Cloning https://github.com/SKTBrain/KoBERT.git to /tmp/pip-install-vi5nnknk/kobert-tokenizer_72ed52d7a7544d9d802fa8af2142b890
  Running command git clone --filter=blob:none --quiet https://github.com/SKTBrain/KoBERT.git /tmp/pip-install-vi5nnknk/kobert-tokenizer_72ed52d7a7544d9d802fa8af2142b890
  Resolved https://github.com/SKTBrain/KoBERT.git to commit fcd729f2f4b37858f333597c0782388ada51eb5f
  Preparing metadata (setup.py) ... done
===== 1. 라이브러리 설치 완료 =====

===== 2. 라이브러리 임포트 및 GPU 설정 =====
Using device: cpu

===== 3. 데이터 로드 및 전처리 =====
로드된 샘플 수: 10000
총 레이블 개수: 7
Train 샘플: 8000, Test 샘플: 2000

===== 4. 토크나이저 및 데이터셋 정의 =====


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/371k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

...

Val Loss: 0.4269, Val Accuracy: 86.20%
--- Epoch 3/3 ---


[Train]:   0%|          | 0/250 [00:00<?, ?it/s]

Train Loss: 0.2495


[Eval]:   0%|          | 0/63 [00:00<?, ?it/s]

Val Loss: 0.3979, Val Accuracy: 87.70%
===== 학습 완료 =====
*** 최종 테스트 정확도: 87.70% ***
